# Energy Spectrum of selected gamma-rays
## Import packages, Load data

In [ ]:
# Imports
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
# Global variables
homePath = "/home/users/l/lebrun/"
dataPath = homePath + "data/"
figPath = homePath + "figures/"
filename = dataPath + "DmlNtup_Data_8years_FullPhotonData_AllSky.root"

In [ ]:
# Load data

tree = None
f = ROOT.TFile(filename, "READ") 

tree = f.Get("DmlNtup")
print(tree)
print(type(tree))

## Inspect energy-related entries

In [ ]:
df = ROOT.RDataFrame(tree)
hE = df.Histo1D(("Energy", "Energy;Energy [GeV];Counts", 800, 1, 110), "tt_bgoTotalE_GeV")

## Plot in Log scale

In [ ]:
ROOT.gStyle.SetOptStat(0)
dfLog = df.Define("log10_E", "TMath::Log10(tt_bgoTotalE_GeV)")
c2 = ROOT.TCanvas()
leg = ROOT.TLegend(0.5, 0.7, 0.85, 0.85)
leg.SetTextSize(0.03)

leg.AddEntry(hE.GetPtr(), "Total Reconstructed Energy", "lep")
leg.SetTextSize(0.03)
hE.SetLineColor(4)
hE.Draw()
leg.Draw("SAME")
c2.SetGrid()
c2.SetLogx()
c2.SaveAs(figPath+"spectrumAll.pdf")
c2.Draw()


## Fit Reco vs corrected energy

### Plot reco vs corr

In [ ]:
hEcorr = df.Histo1D(("Energycorr", "Energy;Energy [GeV];Counts", 800, 1, 110), "tt_bgoTotalEcorr_GeV")

cboth = ROOT.TCanvas()
leg = ROOT.TLegend(0.5, 0.7, 0.85, 0.85)
leg.SetTextSize(0.03)

leg.AddEntry(hE.GetPtr(), "Total Reconstructed Energy", "lep")
leg.AddEntry(hEcorr.GetPtr(), "Total Corrected Energy", "lep")
hE.SetLineColor(4)
hE.Draw()
hEcorr.SetLineColor(3)
hEcorr.Draw("SAME")
leg.Draw("SAME")
cboth.SetGrid()
cboth.SetLogx()
cboth.SetLeftMargin(0.12)
cboth.SaveAs(figPath+"spectrumRecoVsCorr.pdf")
cboth.Draw()


### TGraph opposing the 2

In [ ]:
# Convert DataFrame columns to numpy arrays for easier handling
energy_values = df.AsNumpy(["tt_bgoTotalE_GeV"])["tt_bgoTotalE_GeV"]
corr_values = df.AsNumpy(["tt_bgoTotalEcorr_GeV"])["tt_bgoTotalEcorr_GeV"]

print(energy_values.shape, corr_values.shape)


In [ ]:

# Create TGraph with actual data points
n_points = len(energy_values)
graph = ROOT.TGraph(n_points)

for i in range(n_points):
    graph.SetPoint(i, energy_values[i], corr_values[i])

# Fit a linear function (y = p0 + p1 * x)
fit_func = ROOT.TF1("fit_func", "[0] + [1]*x", min(energy_values), max(energy_values))
graph.Fit(fit_func, "S")

alpha = fit_func.GetParameter(1)
beta = fit_func.GetParameter(0)

leg = ROOT.TLegend(0.15, 0.7, 0.68, 0.85)
leg.SetTextSize(0.05)

leg.AddEntry(fit_func, f"y = {alpha:.3f} x + {beta:.3f}")

In [ ]:
# Draw the results
ROOT.gStyle.SetOptStat(0)
canvas = ROOT.TCanvas("c", "Comparison of Energy vs Other Variable", 800, 600)
graph.SetTitle("Reconstructed Energy vs Corrected Energy;Energy [GeV];Corrected Energy [GeV]")
graph.SetMarkerStyle(20)
graph.GetYaxis().SetTitleOffset(1.4)  # Reduce this value to bring the label closer
graph.Draw("AP")  # 'A' for axes, 'P' for points
fit_func.Draw("SAME")  # Draw fit function
leg.Draw("SAME")
canvas.SetLogx()
canvas.SetLogy()
canvas.SaveAs(figPath+"RecoVsCorr.C")

## Fit of the tail 

In [ ]:
ROOT.gStyle.SetOptStat(1)
l_bound = 1
dfFil = dfLog.Filter(f"tt_bgoTotalE_GeV >= {l_bound}")
hTail = dfFil.Histo1D(("Tail", "Energy [E > 1 GeV];Energy [GeV];Counts", 800, l_bound, 110), "tt_bgoTotalE_GeV")
# Close to 2 - 100 GeV range

In [ ]:
%%cpp
double powerLaw(double *x, double a, double b){
    return b*pow(x[0], a);
}

In [ ]:
%%cpp
double fitFunction(double *x, double *par){
    return powerLaw(x, par[0], par[1]);
}

In [ ]:
ROOT.gStyle.SetOptStat(0)

c3 = ROOT.TCanvas()

fitFuncE = ROOT.TF1("fitFunction", ROOT.fitFunction, xmin=3, xmax=100, npar=2)
fitFuncE.SetParameter(0, -2.5) # a
fitFuncE.SetParameter(1, 4) # b 

res = hTail.Fit(fitFuncE, "R")

fitFuncE.SetRange(3, 100)

alpha = fitFuncE.GetParameter(0)
beta = np.log10(fitFuncE.GetParameter(1))

leg = ROOT.TLegend(0.5, 0.7, 0.85, 0.85)

leg.AddEntry(hTail.GetPtr(), "Total Reconstructed Energy", "lep")
leg.AddEntry(fitFuncE, f"log_{{10}}(y) = {alpha:.2f}log_{{10}}(x) + {beta:.2f}")

hTail.SetLineColor(4)
hTail.Draw()

leg.SetTextSize(0.03)
leg.Draw()
c3.SetLogx()
c3.SetLogy()
c3.SetGrid()
c3.SaveAs(figPath+"energySpectrumTail.pdf")
c3.Draw()

## Divide by acceptance

In [ ]:
# Load acceptance       
acceptance = np.loadtxt(dataPath+"acceptance_vs_energy_cutBDT_v25.txt") # 100 values

ene = np.logspace(1, 5, 101) / 10 

from scipy import interpolate

def create_acceptance_function(energy_edges, acceptance_values):
    """
    Creates an interpolation function that returns acceptance values
    for given energy inputs.
    
    Parameters:
    energy_values (array): Array of energy values in log scale
    acceptance_values (array): Array of corresponding acceptance values
    
    Returns:
    function: Interpolation function that takes energy as input and returns acceptance
    """
    # Calculate bin centers (geometric mean for log-spaced bins)
    energy_centers = np.sqrt(energy_edges[:-1] * energy_edges[1:])
    # Create interpolation function
    # We use 'linear' interpolation which works well for log-scaled data
    interp_func = interpolate.interp1d(
        energy_centers,
        acceptance_values,
        kind='linear',
        bounds_error=False,  # Don't raise error for out-of-bounds values
        fill_value=(acceptance_values[0], acceptance_values[-1])  # Extrapolate using edge values
    )
    
    return interp_func

# Create the function
acceptance_function = create_acceptance_function(ene, acceptance)  # Note: ene has 101 values but acceptance has 100

fig, ax = plt.subplots()
ax.semilogx(ene[:-1], acceptance, "r.-")
ax.set_xlabel("Energy [GeV]")
ax.set_ylabel("Effective acceptance [cm² sr]")
ax.set_title("Effective Acceptance (8 years)")
ax.grid()
fig.savefig(figPath+"acceptance.pdf")

In [ ]:
# Filter from lbound to 1e5

l_bound = 2 # GeV

dfAcc = dfLog.Filter("log10_E < 5").Filter(f"tt_bgoTotalE_GeV >= {l_bound}")

dfNpy = dfAcc.AsNumpy(["tt_bgoTotalE_GeV"])
energies = dfNpy["tt_bgoTotalE_GeV"]
acceptances = acceptance_function(energies)

dfNpy["acceptance"] = acceptances
dfNpy["acc_weight"] = 1/acceptances
dfNpy["energyCorr"] = energies/acceptances

dfAcc = ROOT.RDF.FromNumpy(dfNpy)
dfAcc = dfAcc.Define("log10E_Corr", "TMath::Log10(energyCorr)")

hAcc = dfAcc.Histo1D(("Ecorr", f"Energy [E > {l_bound} GeV];Energy [GeV];Counts/Acceptance", 800, l_bound, 110), "tt_bgoTotalE_GeV", "acc_weight")
fitFuncE = ROOT.TF1("fitFunction", ROOT.fitFunction, xmin=3, xmax=100, npar=2)
fitFuncE.SetParameter(0, -2.5) # a
fitFuncE.SetParameter(1, 4) # b 

res = hAcc.Fit(fitFuncE, "R")

In [ ]:
c = ROOT.TCanvas()
hAcc.Draw()

alpha = fitFuncE.GetParameter(0)
beta = np.log10(fitFuncE.GetParameter(1))

leg = ROOT.TLegend(0.5, 0.7, 0.85, 0.85)

leg.AddEntry(hAcc.GetPtr(), "Total Reconstructed Energy", "lep")
leg.AddEntry(fitFuncE, f"log_{{10}}(y) = {alpha:.2f}log_{{10}}(x) + {beta:.2f}")

leg.SetTextSize(0.03)
leg.Draw()
c.SetLogx()
c.SetLogy()
c.SetGrid()
c.SaveAs(figPath+"energySpectrumCorrected.pdf")
c.Draw()

## Fit again

In [ ]:
%%cpp 
double logPowerLaw(double *x, double alpha, double beta){
    return pow(10, alpha*x[0]) * beta ;
}

In [ ]:
%%cpp 
double fitFunction2(double *x, double *par){
    return logPowerLaw(x, par[0], par[1]);
}

In [ ]:
def filter_by_x_range(histogram, min_x, max_x, string=""):
    # Count the number of bins within the specified range
    valid_bins = 0
    for bin in range(1, histogram.GetNbinsX() + 1):
        bin_center = histogram.GetBinCenter(bin)
        if min_x <= bin_center <= max_x:
            valid_bins += 1
    
    # If no bins in range, return None or an empty histogram
    if valid_bins == 0:
        return None
    
    # Create a new histogram with only the bins in the specified range
    filtered_hist = ROOT.TH1F(
        f"{histogram.GetName()}_filtered", 
        string, 
        valid_bins, 
        min_x, 
        max_x
    )
    
    # Copy bin contents for bins within the range
    new_bin = 1
    for bin in range(1, histogram.GetNbinsX() + 1):
        bin_center = histogram.GetBinCenter(bin)
        bin_content = histogram.GetBinContent(bin)
        
        if min_x <= bin_center <= max_x:
            filtered_hist.SetBinContent(new_bin, bin_content)
            new_bin += 1
    
    return filtered_hist

In [ ]:
fitFuncE2 = ROOT.TF1("fitFunction2", ROOT.fitFunction2, xmin=2, xmax=3, npar=2)
fitFuncE2.SetParameter(0, -2.5) # a
fitFuncE2.SetParameter(1, 4) # b

hEAccFit = filter_by_x_range(hEAcc, 2, 3, "Energy;Energy "+r"[\log_{{10}}(GeV)]"+"; Flux [Counts/Acceptance]")
res2 = hEAccFit.Fit(fitFuncE2, "R")

alpha = fitFuncE2.GetParameter(0)
beta = np.log10(fitFuncE2.GetParameter(1))

leg = ROOT.TLegend(0.4, 0.7, 0.895, 0.85)

leg.AddEntry(hTail.GetPtr(), "Total Reconstructed Energy", "lep")
leg.AddEntry(fitFuncE, rf"\log_{{10}}(y) = {alpha:.2f}\,x + \text{{cst}}")

c4 = ROOT.TCanvas()
hEAccFit.SetTitle("Energy/Acceptance")
hEAccFit.SetXTitle(r"\text{Energy[}\log_{{10}}\text{(GeV)]}")
hEAccFit.SetYTitle(r"\text{Flux} [cm^{-2}sr^{-1}]") # TODO check unit
hEAccFit.Draw()
leg.Draw("same")
c4.SetLogy()
c4.SetGrid()
c4.SaveAs(figPath+"FluxTail.png")
c4.Draw()